# Phase 4 - SQLite analytical SQL

This notebook rebuilds the SQLite database, executes all named SQL analyses, and independently reconciles core KPIs in Python.

In [1]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
for module in ['src.build_database', 'src.run_sql', 'src.reconcile']:
    subprocess.run([sys.executable, '-m', module], cwd=project_root, check=True, stdout=subprocess.DEVNULL)
build = json.loads((project_root / 'reports/generated/sqlite_build_audit.json').read_text(encoding='utf-8'))
manifest = json.loads((project_root / 'reports/generated/sql_query_manifest.json').read_text(encoding='utf-8'))
reconciliation = json.loads((project_root / 'reports/generated/python_sql_reconciliation.json').read_text(encoding='utf-8'))

In [2]:
{key: build[key] for key in ['sqlite_version', 'fact_rows', 'region_rows', 'integrity_check', 'foreign_key_errors']}

{'sqlite_version': '3.53.1',
 'fact_rows': 15956,
 'region_rows': 8,
 'integrity_check': 'ok',
 'foreign_key_errors': 0}

In [3]:
pd.DataFrame(manifest['queries'])[['name', 'rows', 'sql_file']]

,name,rows,sql_file
0,01_dataset_overview,1,sql/kpis.sql
1,02_overall_speed_kpis,1,sql/kpis.sql
2,03_overall_latency_kpis,1,sql/kpis.sql
3,04_loaded_latency_kpis,1,sql/kpis.sql
4,05_quality_control_totals,1,sql/kpis.sql
5,06_quarter_coverage,1,sql/kpis.sql
6,07_regional_measurement_volume,8,sql/regional_analysis.sql
7,08_regional_weighted_performance,8,sql/regional_analysis.sql
8,09_regional_weighted_unweighted_gap,8,sql/regional_analysis.sql
9,10_regional_rankings,8,sql/regional_analysis.sql


In [4]:
pd.read_csv(project_root / 'reports/generated/sql/08_regional_weighted_performance.csv')

,adm1_name,weighted_download_mbps,weighted_upload_mbps,weighted_idle_latency_ms,denominator_tests
0,Dhaka,33.797656,15.061621,26.380619,25540
1,Rajshani,34.166719,13.874831,43.252231,4706
2,Barisal,35.049996,14.652451,32.151846,3385
3,Khulna,35.797405,15.663153,30.637402,4203
4,Mymensingh,38.349168,15.411994,28.628947,1900
5,Rangpur,42.503375,15.057190,39.913987,4976
6,Chittagong,42.783097,18.118409,33.092087,13813
7,Sylhet,45.496427,20.453897,30.496715,4262


In [5]:
pd.DataFrame(reconciliation['comparisons'])

,metric,python,sqlite,difference,tolerance,passed
0,tiles,15956.000000,15956.000000,0.000000e+00,1.000000e-09,True
1,tests,62785.000000,62785.000000,0.000000e+00,1.000000e-09,True
2,devices,29693.000000,29693.000000,0.000000e+00,1.000000e-09,True
3,tile_avg_download_mbps,36.078125,36.078125,0.000000e+00,1.000000e-09,True
4,test_weighted_download_mbps,37.625394,37.625394,0.000000e+00,1.000000e-09,True
5,test_weighted_upload_mbps,16.039675,16.039675,3.552714e-15,1.000000e-09,True
6,test_weighted_idle_latency_ms,31.137915,31.137915,0.000000e+00,1.000000e-09,True
7,complete_loaded_latency_tiles,15483.000000,15483.000000,0.000000e+00,1.000000e-09,True


## Interpretation

Equivalent Python and SQLite populations reconcile. Weighted values are reconstructed from rounded tile averages and are not raw-test statistics. Regional comparisons remain descriptive and must carry measurement-volume caveats.